# Best TSAD - Train SOTA Models on Kaggle

This notebook trains AER and Anomaly Transformer models on Kaggle's free GPU and saves the trained models.

**Setup:**
1. Upload your AnomLLM data to Kaggle Dataset
2. Enable GPU: Settings → Accelerator → GPU T4 x2
3. Run all cells
4. Download trained models from Output

In [ ]:
# Install dependencies
!pip install -q scikit-learn pyyaml pandas

In [ ]:
# Create directory structure
!mkdir -p src/models src/pipeline src/data src/utils src/evaluation configs/pipelines

## Upload Best-TSAD Code

Upload the following files from your local `best-tsad` folder:
- All files from `src/models/`
- All files from `src/pipeline/`  
- All files from `src/data/`
- All files from `src/utils/`
- All files from `src/evaluation/`
- Config files from `configs/pipelines/`

Or use the code cells below to copy-paste the essential code.

## Option 1: Upload as Kaggle Dataset

**Easiest method:**
1. Zip your entire `best-tsad` folder
2. Go to Kaggle → Datasets → New Dataset
3. Upload the zip file
4. In this notebook: Add Dataset → Your uploaded dataset
5. Run: `!unzip ../input/best-tsad/best-tsad.zip`

## Option 2: Clone from GitHub (if you pushed code)

If you've pushed your code to GitHub:

In [ ]:
# !git clone https://github.com/yourusername/best-tsad.git
# %cd best-tsad

## Load AnomLLM Data

**Setup:**
1. Upload `src/data/synthetic` folder as Kaggle Dataset
2. Add it to this notebook
3. Update the path below

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/best-tsad')  # Adjust if needed

# Import after uploading code
from src.data.anomllm_loader import load_anomllm_category
from src.utils.config_factory import load_config, build_pipeline_from_config
from src.evaluation.evaluator import Evaluator

print("Imports successful!")

In [ ]:
# Load one category for testing
CATEGORY = 'point'

print(f"Loading {CATEGORY} category...")
dataset = load_anomllm_category(CATEGORY, base_path="/kaggle/input/anomllm-synthetic")

print(f"Train size: {len(dataset.X_train)}")
print(f"Test size: {len(dataset.X_test)}")
print(f"Test anomalies: {dataset.y_test.sum()}")

## Train AER Model

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Load AER config (modify for GPU)
config = load_config("/kaggle/working/best-tsad/configs/pipelines/aer_pipeline.yaml")

# Override for GPU and better performance
config['data_processing']['params']['device'] = 'cuda'
config['data_processing']['params']['hidden_dim'] = 128
config['data_processing']['params']['epochs'] = 50

print("Building AER pipeline...")
pipeline = build_pipeline_from_config(config)

In [ ]:
# Sample data for faster training (adjust as needed)
X_train_sample = dataset.X_train[:50000]
y_train_sample = dataset.y_train[:50000]
X_test_sample = dataset.X_test[:30000]
y_test_sample = dataset.y_test[:30000]

print(f"Training on {len(X_train_sample)} samples...")
pipeline.fit(X_train_sample, y_train_sample)

In [ ]:
# Test
print("Testing...")
result = pipeline.predict(X_test_sample, y_test_sample)

# Evaluate
evaluator = Evaluator()
eval_result = evaluator.evaluate(y_test_sample, result.predictions, result.point_scores)

print("\n" + "="*50)
print("AER RESULTS")
print("="*50)
print(f"F1 Score:    {eval_result.f1:.3f}")
print(f"Precision:   {eval_result.precision:.3f}")
print(f"Recall:      {eval_result.recall:.3f}")
print(f"PA-F1:       {eval_result.pa_f1:.3f}")
print("="*50)

## Save AER Model

In [ ]:
import torch
import pickle

# Save model weights
torch.save(
    pipeline.data_processor.model.state_dict(),
    f'/kaggle/working/aer_model_{CATEGORY}.pth'
)

# Save metadata
metadata = {
    'category': CATEGORY,
    'window_size': config['data_processing']['window_size'],
    'hidden_dim': config['data_processing']['params']['hidden_dim'],
    'num_layers': config['data_processing']['params']['num_layers'],
    'input_dim': pipeline.data_processor.input_dim,
    'f1': eval_result.f1,
    'pa_f1': eval_result.pa_f1
}

with open(f'/kaggle/working/aer_model_{CATEGORY}_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print(f"AER model saved to /kaggle/working/aer_model_{CATEGORY}.pth")
print("Download from Output tab after running!")

## Train Anomaly Transformer

Repeat the same process for Anomaly Transformer:

In [ ]:
# Load Transformer config
config_transformer = load_config("/kaggle/working/best-tsad/configs/pipelines/transformer_pipeline.yaml")

# Override for GPU
config_transformer['data_processing']['params']['device'] = 'cuda'
config_transformer['data_processing']['params']['d_model'] = 512

print("Building Anomaly Transformer pipeline...")
pipeline_transformer = build_pipeline_from_config(config_transformer)

In [ ]:
print(f"Training Transformer on {len(X_train_sample)} samples...")
pipeline_transformer.fit(X_train_sample, y_train_sample)

In [ ]:
# Test Transformer
print("Testing Transformer...")
result_transformer = pipeline_transformer.predict(X_test_sample, y_test_sample)

eval_result_transformer = evaluator.evaluate(
    y_test_sample,
    result_transformer.predictions,
    result_transformer.point_scores
)

print("\n" + "="*50)
print("ANOMALY TRANSFORMER RESULTS")
print("="*50)
print(f"F1 Score:    {eval_result_transformer.f1:.3f}")
print(f"Precision:   {eval_result_transformer.precision:.3f}")
print(f"Recall:      {eval_result_transformer.recall:.3f}")
print(f"PA-F1:       {eval_result_transformer.pa_f1:.3f}")
print("="*50)

In [ ]:
# Save Transformer model
torch.save(
    pipeline_transformer.data_processor.model.state_dict(),
    f'/kaggle/working/transformer_model_{CATEGORY}.pth'
)

metadata_transformer = {
    'category': CATEGORY,
    'window_size': config_transformer['data_processing']['window_size'],
    'd_model': config_transformer['data_processing']['params']['d_model'],
    'n_heads': config_transformer['data_processing']['params']['n_heads'],
    'n_layers': config_transformer['data_processing']['params']['n_layers'],
    'input_dim': pipeline_transformer.data_processor.input_dim,
    'f1': eval_result_transformer.f1,
    'pa_f1': eval_result_transformer.pa_f1
}

with open(f'/kaggle/working/transformer_model_{CATEGORY}_metadata.pkl', 'wb') as f:
    pickle.dump(metadata_transformer, f)

print(f"Transformer model saved to /kaggle/working/transformer_model_{CATEGORY}.pth")

## Compare Results

In [ ]:
import pandas as pd

results_comparison = pd.DataFrame([
    {
        'Model': 'AER',
        'F1': eval_result.f1,
        'Precision': eval_result.precision,
        'Recall': eval_result.recall,
        'PA-F1': eval_result.pa_f1
    },
    {
        'Model': 'Anomaly Transformer',
        'F1': eval_result_transformer.f1,
        'Precision': eval_result_transformer.precision,
        'Recall': eval_result_transformer.recall,
        'PA-F1': eval_result_transformer.pa_f1
    }
])

print("\nCOMPARISON:")
print(results_comparison.to_string(index=False))

# Save comparison
results_comparison.to_csv(f'/kaggle/working/comparison_{CATEGORY}.csv', index=False)
print(f"\nComparison saved to /kaggle/working/comparison_{CATEGORY}.csv")

## Download Models

After running all cells:
1. Go to Output tab (right sidebar)
2. Download the `.pth` and `.pkl` files
3. Copy to your local `best-tsad/models/` folder

**Files to download:**
- `aer_model_{category}.pth`
- `aer_model_{category}_metadata.pkl`
- `transformer_model_{category}.pth`
- `transformer_model_{category}_metadata.pkl`
- `comparison_{category}.csv`

## Load Models Locally

After downloading, use this code locally to load the trained models:

In [ ]:
# LOCAL CODE (run on your machine)
'''
import torch
import pickle
from src.models.aer import AERModel

# Load metadata
with open('models/aer_model_point_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

# Recreate model
model = AERModel(
    input_dim=metadata['input_dim'],
    hidden_dim=metadata['hidden_dim'],
    num_layers=metadata['num_layers']
)

# Load weights
model.load_state_dict(torch.load('models/aer_model_point.pth'))
model.eval()

print(f"Model loaded! F1: {metadata['f1']:.3f}")
'''